[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Reading and Writing Text


## What you will be able to do

Open a file in the right mode for what you are doing, and explain why a file you have written
can show as empty until you close it. You will also be able to say what `with` does beyond
saving a line.


## The idea

### The problem

The **Paths** notebook found the file. This one opens it, and the opening is where a surprising
amount goes wrong.

Writing to a file does not necessarily put anything on the disk. Check the size straight after
writing and it can be zero, with no error and nothing to suggest the data went anywhere. That is
not a bug and it is not rare: it is how file writing works everywhere, and it is why a program
killed halfway through leaves a file that is empty rather than half-written.

There is a second problem that costs more. Opening a file in the wrong mode does not fail. It
does something, and the something is occasionally destructive. One wrong character between `"r"`
and `"w"` deletes the file's contents before your code runs a single line.

### What open gives you

> `open(path, mode)` returns a **file object**: a connection to the file's contents, positioned
> at a particular point, in a particular mode. The mode decides whether you may read, whether
> you may write, and what happens to what was already there.
>
> A file object holds an operating system resource and must be closed. `with` closes it for you,
> including when the code inside raises.

### The modes

Three basic ones, and one worth knowing about.

| Mode | Reads | Writes | If the file exists | If it does not |
|---|---|---|---|---|
| `"r"` | yes | no | opens at the start | `FileNotFoundError` |
| `"w"` | no | yes | **empties it immediately** | creates it |
| `"a"` | no | yes | opens at the end | creates it |
| `"x"` | no | yes | `FileExistsError` | creates it |

`"r"` is the default, which is why `open(path)` is safe.

`"x"` deserves more use than it gets. It says "create this, and fail loudly if something is
already there", which is exactly right when you are writing a result you do not want to
overwrite by accident.

### Buffering

Writing goes into memory first. The operating system moves it to the disk later, in batches,
because writing one large block is far faster than writing a thousand small ones.

That is normally invisible. It becomes visible in three situations, all of which look like
bugs:

- You write, then check the file, and it is empty.
- Another program reads the file while yours is still writing, and sees nothing.
- Your program is killed partway through, and the file is empty rather than partly written.

Closing the file flushes the buffer. `with` closes the file. That is most of why `with` is not
optional.

### Where you will meet this

Every remaining notebook in this guide opens files, and the ones about CSV, JSON and Excel are
all built on this. The mode question comes back every time you write a result.

### What this notebook covers

- The four modes, and the one that destroys before your code runs
- Buffering, measured on the disk rather than described
- `with`, and what it does when the block raises
- Reading whole, by line, and in chunks
- `writelines`, which adds nothing you did not give it
- Text mode against binary mode, and the newline translation you did not ask for
- Where you are in the file: `tell` and `seek`
- Three errors, plus one that empties a file without raising

### A first look

Nothing to run yet.

```python
f = open("notes.txt", "w")
f.write("some text")

print(Path("notes.txt").stat().st_size)   # 0

f.close()

print(Path("notes.txt").stat().st_size)   # 9
```

The text was written, and the file was empty until it was closed. That gap is the subject of
half this notebook, and `with` is how you stop caring about it.


## Setup

Two imports and a folder to work in.

- `Path` builds paths and checks file sizes on disk
- `shutil` removes the scratch folder at the end

**Run this cell before the rest of the notebook.**


In [1]:
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

print("working in:", scratch, "->", scratch.exists())


working in: scratch -> True


## Worked examples

### Buffering, measured

This is the promise from the **Paths** notebook, and it is worth watching rather than reading.


In [2]:
note = scratch / "buffered.txt"

f = open(note, "w")
f.write("this text has been written\n")
print("after write:", note.stat().st_size, "bytes on disk")

f.flush()
print("after flush:", note.stat().st_size, "bytes on disk")

f.close()
print("after close:", note.stat().st_size, "bytes on disk")


after write: 0 bytes on disk
after flush: 27 bytes on disk
after close: 27 bytes on disk


Zero, then 27, then 27. The `write` put the text in a buffer in memory. `flush` pushed it to the
operating system. `close` flushes as well, which is why the last number would have been right
even without the explicit flush.

Nothing failed at any point. A program that wrote the file and crashed before closing it would
have left an empty file and no indication why.

### with, which closes for you


In [3]:
note = scratch / "withblock.txt"

with open(note, "w") as f:
    f.write("written inside the block\n")
    print("inside the block:", note.stat().st_size, "bytes")

print("after the block: ", note.stat().st_size, "bytes")


inside the block: 0 bytes
after the block:  25 bytes


Same behavior, and nothing to remember. The file is closed when the block ends.

The part that matters more is what happens when the block does not end normally.


In [4]:
handle = {}

try:
    with open(scratch / "withblock.txt") as f:
        handle["f"] = f
        raise ValueError("something went wrong in the middle")
except ValueError as e:
    print("the exception still escaped:", e)

print("but the file was closed:", handle["f"].closed)


the exception still escaped: something went wrong in the middle
but the file was closed: True


The exception propagated normally, and the file was closed on the way out. Written by hand, the
`close()` after the failing line would never have run, which is the `finally` behavior the
**Errors and Exceptions** notebook covered, applied automatically.

An open file that is never closed holds an operating system handle. One does not matter. A loop
over ten thousand files that forgets to close them runs out of handles and fails with an error
that says nothing about the real cause.


### The modes, tested

Rather than trusting the table above, here it is run.


In [5]:
(scratch / "exists.txt").write_text("original contents\n")

for mode, name in [("r", "exists.txt"), ("a", "exists.txt"),
                   ("x", "exists.txt"), ("x", "brandnew.txt"),
                   ("r", "missing.txt")]:
    try:
        with open(scratch / name, mode):
            print(f"open({name:<13} {mode!r}) -> ok")
    except Exception as e:
        print(f"open({name:<13} {mode!r}) -> {type(e).__name__}")


open(exists.txt    'r') -> ok
open(exists.txt    'a') -> ok
open(exists.txt    'x') -> FileExistsError
open(brandnew.txt  'x') -> ok
open(missing.txt   'r') -> FileNotFoundError


`"w"` is deliberately missing from that list, because running it would have emptied
`exists.txt` before anything else could be demonstrated. Common errors shows it doing exactly
that.

`"x"` is the one worth adopting. When a script writes a result file, `"x"` turns "silently
overwrote yesterday's output" into an error you can see.


### Reading: whole, by line, in chunks

Three ways, for three sizes of file.


In [6]:
three = scratch / "three.txt"
three.write_text("one\ntwo\nthree\n")

with open(three) as f:
    print("read():     ", repr(f.read()))


read():      'one\ntwo\nthree\n'


`read()` gives the whole file as one string. Fine for anything that fits in memory comfortably,
and a problem for a file larger than your memory.


In [7]:
with open(three) as f:
    print("readline():", repr(f.readline()))
    print("readline():", repr(f.readline()))

with open(three) as f:
    print("readlines():", f.readlines())


readline(): 'one\n'
readline(): 'two\n'
readlines(): ['one\n', 'two\n', 'three\n']


`readline()` gives one line and moves forward. `readlines()` gives all of them as a list, which
is `read()` with extra steps and the same memory cost.

Looping over the file object is the form to use. It reads one line at a time and holds only that
line.


In [8]:
with open(three) as f:
    for number, line in enumerate(f, start=1):
        print(number, repr(line))


1 'one\n'
2 'two\n'
3 'three\n'


For a file with no lines in it, such as a large binary or one enormous line, read fixed-size
chunks instead.


In [9]:
big = scratch / "big.txt"
big.write_text("x" * 5000)

total = 0
with open(big) as f:
    while chunk := f.read(1024):
        total += len(chunk)

print("read", total, "characters in chunks of 1024")


read 5000 characters in chunks of 1024


`while chunk := f.read(1024)` reads, assigns and tests in one line. `read` returns an empty
string at the end of the file, which is falsy, so the loop stops. That `:=` is the operator the
**Booleans and Comparison** notebook told you to ignore; this is the case it exists for.


### Writing, and writelines

`write` takes one string and adds nothing to it.


In [10]:
out = scratch / "written.txt"

with open(out, "w") as f:
    f.write("first")
    f.write("second")

print(repr(out.read_text()))


'firstsecond'


`firstsecond`, on one line. `print` adds a newline and `write` does not, which catches everyone
once.


In [11]:
with open(out, "w") as f:
    f.write("first\n")
    f.write("second\n")

print(repr(out.read_text()))


'first\nsecond\n'


`writelines` is the same story with a misleading name. It writes a list of strings and adds no
line endings at all.


In [12]:
with open(out, "w") as f:
    f.writelines(["a", "b", "c"])
print("without newlines:", repr(out.read_text()))

with open(out, "w") as f:
    f.writelines(["a\n", "b\n", "c\n"])
print("with newlines:   ", repr(out.read_text()))


without newlines: 'abc'
with newlines:    'a\nb\nc\n'


It does not write lines. It writes strings, one after another, and the name is a historical
accident. `"\n".join(items)` passed to `write` is clearer about what it does.


### Text mode against binary mode

Adding `"b"` to the mode gives you the bytes on the disk rather than decoded text.


In [13]:
accented = scratch / "accented.txt"
accented.write_text("h\u00e9llo\n")

with open(accented) as f:
    print("text:  ", repr(f.read()))

with open(accented, "rb") as f:
    print("binary:", repr(f.read()))


text:   'héllo\n'
binary: b'h\xc3\xa9llo\n'


One accented character became two bytes. That translation is the subject of the **Encodings**
notebook, which is next.

Text mode also rewrites line endings. A file written on Windows ends its lines with `\r\n`, and
text mode hides that from you.


In [14]:
windows = scratch / "windows.txt"
windows.write_bytes(b"one\r\ntwo\r\n")

with open(windows) as f:
    print("default:    ", repr(f.read()))

with open(windows, newline="") as f:
    print("newline='':", repr(f.read()))

with open(windows, "rb") as f:
    print("binary:     ", repr(f.read()))


default:     'one\ntwo\n'
newline='': 'one\r\ntwo\r\n'
binary:      b'one\r\ntwo\r\n'


The default is usually what you want, and it is why a file from a Windows machine reads
correctly without any effort.

It matters when you are writing a format where the exact bytes are specified. The **CSV**
notebook uses `newline=""` for exactly that reason, and explains why there.


### Where you are in the file

A file object remembers a position, and reading moves it.


In [15]:
with open(three) as f:
    print("at start:      ", f.tell())
    f.read(3)
    print("after read(3): ", f.tell())
    f.seek(0)
    print("after seek(0): ", f.tell())
    print("next line:     ", repr(f.readline()))


at start:       0
after read(3):  3
after seek(0):  0
next line:      'one\n'


That position is why reading a file twice gives nothing the second time, which the
**Files and Paths** notebook showed. `seek(0)` rewinds.

You will rarely need `seek` for text. It matters for binary formats where you jump to a known
offset, and it is worth knowing the position exists.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/02-reading-and-writing-text-solutions.ipynb).

**1.** Write three lines to `scratch/notes.txt` using `with` and `write`, then print the file
back. Make sure the three lines are actually three lines.


In [16]:
# your code here


**2.** Append a fourth line without losing the first three, then print the whole file.


In [17]:
# your code here


**3.** Open `scratch/notes.txt` with mode `"x"` and print the exception type rather than letting
it stop the cell.


In [18]:
# your code here


**4.** Loop over the file printing each line numbered, with the trailing newline removed.


In [19]:
# your code here


**5.** Open a file, write to it, and check the size on disk before and after closing, without
using `with`. Say in a comment which line made the data appear.


In [20]:
# your code here


**6.** Write `["red", "green", "blue"]` to a file so that each appears on its own line, using
`writelines`. Then do the same with `write` and a join, and confirm the files match.


In [21]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### The quiet one: "w" empties the file before you write anything

This does not raise, and it is the most expensive mistake in this notebook.


In [22]:
important = scratch / "important.txt"
important.write_text("months of results\n")

print("before:", repr(important.read_text()))

with open(important, "w") as f:
    pass                       # opened, wrote nothing, closed

print("after: ", repr(important.read_text()))


before: 'months of results\n'
after:  ''


The file is empty. Nothing was written, nothing failed, and the contents were destroyed the
instant the file was opened.

`"w"` truncates on open, before your first line of code inside the block runs. Typing `"w"` when
you meant `"r"` is one keystroke, and there is no undo.

Two habits that prevent it:

- Use `"x"` when creating something new, so an existing file is an error rather than a casualty.
- Use `"a"` when adding, and reserve `"w"` for files you are genuinely regenerating.

The **Writing Safely** notebook covers the version used when the data actually matters: write to
a temporary file, then rename it into place.


### FileExistsError: x mode found something there


In [23]:
with open(scratch / "important.txt", "x") as f:
    f.write("this will not happen\n")


FileExistsError: [Errno 17] File exists: 'scratch/important.txt'

`File exists`. This is `"x"` working: it refused rather than overwriting. Compare it with the
cell above, which did the same thing with `"w"` and destroyed the file silently.


### ValueError: the file is closed

Using a file object after its `with` block has ended.


In [24]:
with open(three) as f:
    first = f.readline()

print("read inside the block:", repr(first))
print(f.readline())


read inside the block: 'one\n'


ValueError: I/O operation on closed file.

`I/O operation on closed file`. The name `f` still exists after the block, and the file it
referred to does not.

Do the reading inside the block, or read what you need into a variable, as the first line here
does.


### UnsupportedOperation: reading a file opened for writing


In [25]:
with open(scratch / "notes.txt", "w") as f:
    f.write("hello\n")
    print(f.read())


UnsupportedOperation: not readable

`not readable`. The mode is a promise about what you will do, and Python holds you to it.

`"r+"` and `"w+"` allow both, and they behave differently enough from each other to be worth
looking up when you need them. Most code does not: read the file, close it, write a new one.


### Cleaning up


In [26]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- `"r"` reads, `"w"` **empties then writes**, `"a"` appends, `"x"` creates or fails.
- `"w"` truncates the moment the file is opened, before any of your code runs.
- Writes are buffered, so a file can be empty on disk until it is flushed or closed.
- `with` closes the file, including when the block raises, which is why it is not optional.
- Loop over the file object to read one line at a time; use `read(n)` for fixed chunks.
- `write` and `writelines` add no newlines of their own.
- Text mode decodes bytes and translates line endings; `"rb"` gives you what is actually there.
- A file object remembers a position, so reading it twice gives nothing until you `seek(0)`.


## What is next

The **Encodings** notebook, which explains the translation text mode performs: what UTF-8 is,
what those replacement characters in imported data actually are, and how to read a file whose
encoding you were told wrongly.


---

&#8592; **Previous:** [Paths](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/01-paths.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)  &nbsp;·&nbsp;  **Next:** [Encodings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/03-encodings.ipynb) &#8594;
